# Baseline Compare

Run the shared baseline strategies on one dataset and compare their metrics side by side.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [ ]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


In [ ]:
from pathlib import Path
import pandas as pd

from portfolio_toolkit import baseline_weights, backtest_weights, write_backtest_artifacts

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
baseline_names = ['equal_weight', 'inverse_volatility', 'momentum_20d']
results = []
for name in baseline_names:
    weights = baseline_weights(dataset_name, name, repo_root=repo_root)
    result = backtest_weights(dataset_name, weights, repo_root=repo_root)
    write_backtest_artifacts(result, repo_root / 'runs' / 'baseline_compare' / name)
    row = {'strategy_name': name, **result.metrics}
    results.append(row)

comparison = pd.DataFrame(results).sort_values('sharpe', ascending=False)
comparison

In [ ]:
# Validation cell
assert set(comparison['strategy_name']) == set(baseline_names)
assert {'total_return', 'annual_return', 'sharpe'}.issubset(comparison.columns)
print('Baseline comparison validated.')